# ChebAr on the anb-compute GPU cluster

The original 13-cell version of this notebook ran on `chuc` via OAR, with a local
GPU and `CUDA_VISIBLE_DEVICES` pinned by hand. There is no local GPU any more:
the work runs on the anb-compute Ray cluster, which takes **one picklable
function** and gives back **its return value**. So the whole pipeline
(build -> first estimation -> Chebyshev filter -> Arnoldi -> rate -> residual)
lives in a single function below, and only plotting happens here.

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`).
   Those versions must match the cluster; a 3.14 kernel cannot talk to it.

2. Run all cells.

The first cell mints the cluster token itself, so the kernel does **not** need to
have been started from a shell with `RAY_AUTH_TOKEN` exported. It does need to
run before anything imports `ray`, which in a fresh kernel it does — if you have
already imported ray, restart the kernel.

The first `acr.run` of a session also needs GCS read credentials to fetch its
result, which may open an interactive Microsoft device-code login (a URL and a
code printed below the cell). That is expected, and cached afterwards.

Expect roughly **2–3 minutes**: most of it is the filtered Arnoldi on the GPU,
plus ~20 s of job startup on a warm pool.

In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel. If
# you have already run anything that imports ray, restart the kernel first.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

# Mint the cluster token from Vault here, so the notebook also works from a
# kernel that was not launched from a shell with RAY_AUTH_TOKEN exported (which
# is the normal case in VS Code). `vault login` is interactive, so it stays a
# manual, once-a-week step.
if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import ray

from anb_compute import ray as acr

print("python", sys.version.split()[0], "| ray", ray.__version__)

import pathlib


def _find_repo():
    """Locate the Cheb_Ar checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "cheb_ar").is_dir():
            return base
        if (base / "Cheb_Ar" / "src" / "cheb_ar").is_dir():
            return base / "Cheb_Ar"
    raise RuntimeError(f"no Cheb_Ar checkout found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
CHEB_AR = str(REPO / "src" / "cheb_ar")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (CHEB_AR, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[CHEB_AR, DYNAMIQS])

In [ ]:
def run_cheb_ar(
    n_a=25,
    n_b=11,
    alpha_sq=8.5,
    m_arnoldi_0=60,
    m_arnoldi=120,
    margin=1e-2,
    seed=0,
    cheb_degree=6,
):
    """The full ChebAr pipeline for one parameter point, on one GPU worker.

    Returns plain numpy/python objects only: the client has neither jax nor
    dynamiqs installed, so nothing jax-flavoured may cross back. The Wigner grid
    is therefore computed here too, and the client only plots it.
    """
    import time

    import numpy as np

    import dynamiqs as dq
    from cheb_ar import ChebAr
    from cheb_ar.models.ats import build_ats_hamiltonian_interaction

    def py(v):
        """0-d jax/numpy scalar -> plain python float/complex."""
        a = np.asarray(v)
        return complex(a) if np.iscomplexobj(a) else float(a)

    t0 = time.time()
    dims = (n_a, n_b)
    N = n_a * n_b

    # --- interaction frame: mesolve_fast + output_phase + frame transform V ---
    (
        H_I,
        jump_ops_I,
        jump_ops_LdL_I,
        output_phase,
        V,
        T_block,
        params,
    ) = build_ats_hamiltonian_interaction(n_a=n_a, n_b=n_b, alpha_sq=alpha_sq)

    solver = ChebAr(
        H_I,
        jump_ops_I,
        T_block,
        jump_ops_LdL=jump_ops_LdL_I,
        dims=dims,
        output_phase=output_phase,
        cheb_degree=cheb_degree,
    )
    t_build = time.time() - t0

    # --- 1. unfiltered Arnoldi, to estimate the spectrum ---
    x0 = solver.make_x0(seed=seed)
    _, _, ritz_vals = solver.first_estimation(x0, m_arnoldi=m_arnoldi_0)
    t_est = time.time() - t0

    # --- 2. Chebyshev filter from that estimate, then filtered Arnoldi ---
    solver.setup_chebyshev(ritz_vals, margin=margin)
    Q, H, mu_list = solver.arnoldi_hessenberg(
        x0, solver.chebyshev_filter, m_arnoldi, warm_start=False
    )
    t_arnoldi = time.time() - t0

    # --- 3. bit-flip rate + residual on the true propagator ---
    rate_bf = solver.rate_from_mu(mu_list[-1])
    x_ritz, _ = solver.ritz_vector(Q, H, m_arnoldi)
    res = solver.residual_check(x_ritz)

    # --- 4. back to the lab frame, storage-mode Wigner ---
    V = np.asarray(V)
    rho_lab = V @ np.asarray(x_ritz).reshape(N, N) @ V.conj().T
    rho_a = dq.ptrace(dq.asqarray(rho_lab, dims=dims), 0)
    xvec, yvec, W = dq.wigner(rho_a)

    import jax

    return {
        "rate_bf": py(rate_bf),
        "mu": py(mu_list[-1]),
        "mu_RQ": py(res["mu_RQ"]),
        "rate_RQ": py(res["rate_RQ"]),
        "res_rel": py(res["res_rel"]),
        "ritz_vals": np.asarray(ritz_vals),
        "x_ritz": np.asarray(x_ritz),
        "wigner": (np.asarray(xvec), np.asarray(yvec), np.asarray(W)),
        "params": {k: py(v) for k, v in params.items()},
        "settings": {
            "n_a": n_a,
            "n_b": n_b,
            "alpha_sq": alpha_sq,
            "m_arnoldi_0": m_arnoldi_0,
            "m_arnoldi": m_arnoldi,
            "margin": margin,
            "seed": seed,
            "cheb_degree": cheb_degree,
        },
        "timings_s": {
            "build": t_build,
            "first_estimation": t_est - t_build,
            "arnoldi": t_arnoldi - t_est,
            "total": time.time() - t0,
        },
        "device": str(jax.devices()[0]),
    }

In [ ]:
import time

# num_gpus > 0 routes to the gpu pool (on-demand only) and runs `run_cheb_ar` as
# a task on a worker rather than inline in the driver. Logs print when the job
# ends; follow a long one with `ray job logs --follow <job_id>`.
t0 = time.time()
result = acr.run(
    run_cheb_ar,
    num_gpus=1,
    num_cpus=8,
    runtime_env=RUNTIME_ENV,
    poll_seconds=5,
)
print(f"\nwall clock, incl. job startup: {time.time() - t0:.1f} s")
print("device :", result["device"])
print("timings:", {k: round(v, 1) for k, v in result["timings_s"].items()})

In [ ]:
print("rate_bf :", result["rate_bf"])
print("mu      :", result["mu"])
print("mu_RQ   :", result["mu_RQ"])
print("rate_RQ :", result["rate_RQ"])
print("res_rel :", result["res_rel"])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

xvec, yvec, W = result["wigner"]

# Same convention as dq.plot.wigner: W is indexed [x, y], so transpose for
# pcolormesh, and clip to +-2/pi.
vmax = 2 / np.pi
fig, ax = plt.subplots(figsize=(5.0, 4.2), constrained_layout=True)
im = ax.pcolormesh(
    xvec, yvec, np.clip(W.T, -vmax, vmax),
    cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="auto",
)
ax.set_aspect("equal")
ax.set_xlabel(r"$\mathrm{Re}(\alpha)$")
ax.set_ylabel(r"$\mathrm{Im}(\alpha)$")
ax.set_title(rf"storage mode, $|\alpha|^2 = {result['settings']['alpha_sq']}$")
fig.colorbar(im, ax=ax, ticks=[-vmax, 0, vmax], label="W")
plt.show()